# 🎓 Radar de Evasão Escolar

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Anoterrr/evasao-escolar/blob/main/colab.ipynb)

Trabalho final de **Sistemas de Inteligência Artificial**. Este notebook percorre o projeto passo a passo:

1. **Os dados**: base pública da UCI com 4.424 alunos de graduação
2. **Análise exploratória**: o que separa quem evade de quem fica
3. **Engenharia de atributos**: colunas novas criadas a partir das originais
4. **Treino**: comparação de Regressão Logística, Random Forest e XGBoost
5. **Resultados**: métricas no conjunto de teste
6. **Usando o modelo**: risco de evasão de alunos fictícios
7. **App**: abrir o painel Streamlit a partir do Colab

> Para rodar: **Ambiente de execução → Executar tudo**. Não precisa de GPU.

## 0. Preparação

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("evasao-escolar"):
        !git clone -q https://github.com/Anoterrr/evasao-escolar.git
    %cd evasao-escolar
    # uv cria um ambiente separado com as versões exatas do projeto (uv.lock),
    # sem mexer nas bibliotecas que já vêm no Colab
    !pip install -q uv
    !uv sync -q

print("Rodando no Colab" if IN_COLAB else "Rodando localmente", "| pasta:", os.getcwd())

In [ ]:
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.dados import CURSOS, ROTULOS, carregar_dataset, criar_atributos

# Cores: azul para séries únicas; verde/amarelo/vermelho só para as faixas de risco
AZUL = "#2a78d6"
COR_FAIXA = {"Baixo": "#0ca30c", "Médio": "#fab219", "Alto": "#d03b3b"}
pd.set_option("display.max_columns", 50)

## 1. Os dados

**Predict Students' Dropout and Academic Success**: UCI Machine Learning Repository, ID 697
(Martins, Tolledo, Machado, Baptista e Realinho, 2021, DOI [10.24432/C5MC89](https://doi.org/10.24432/C5MC89)).

Cada linha é um aluno de uma instituição de ensino superior em Portugal. As colunas descrevem o perfil, a situação
financeira e o desempenho no 1º e no 2º semestre. A função `carregar_dataset` traduz os nomes das colunas para
português e cria o alvo **`evadiu`** (1 = Dropout, 0 = Graduate ou Enrolled).

In [ ]:
df = carregar_dataset()
print(f"{df.shape[0]} alunos, {df.shape[1] - 2} atributos de entrada")
print("Valores ausentes:", int(df.isna().sum().sum()))
df.head()

In [ ]:
situacao = (df["situacao"].map({"Dropout": "Evadiu", "Graduate": "Formou-se", "Enrolled": "Matriculado"})
            .value_counts().rename_axis("situação").reset_index(name="alunos"))
situacao["%"] = (situacao["alunos"] / situacao["alunos"].sum() * 100).round(1)
display(situacao)
print(f"Alvo binário -> evadiu: {df['evadiu'].mean():.1%} | não evadiu: {1 - df['evadiu'].mean():.1%}")
print("A base é desbalanceada: há cerca de 2 alunos que ficam para cada aluno que evade.")

## 2. Análise exploratória

Antes de treinar qualquer modelo, vale olhar o que separa quem evade de quem fica.

In [ ]:
fatores = {
    "mensalidades_em_dia": {1: "Mensalidade em dia", 0: "Mensalidade atrasada"},
    "devedor": {0: "Não devedor", 1: "Devedor"},
    "bolsista": {0: "Sem bolsa", 1: "Bolsista"},
}
linhas = []
for col, nomes in fatores.items():
    for valor, nome in nomes.items():
        grupo = df[df[col] == valor]
        linhas.append({"grupo": nome, "alunos": len(grupo), "taxa de evasão (%)": round(grupo["evadiu"].mean() * 100, 1)})
pd.DataFrame(linhas)

**Mensalidade atrasada** é o fator isolado mais forte: a maioria desses alunos evadiu. Mas isso pode ser *sintoma*
(quem já decidiu sair para de pagar), e não causa.

In [ ]:
# Taxa de aprovação no 2º semestre (aprovadas / matriculadas) em faixas
s2 = df[df["s2_matriculadas"] > 0].copy()
s2["taxa_aprov_2s"] = s2["s2_aprovadas"] / s2["s2_matriculadas"]
s2["faixa"] = pd.cut(s2["taxa_aprov_2s"], [-0.01, 0, 0.25, 0.5, 0.75, 0.99, 1],
                     labels=["0%", "1-25%", "26-50%", "51-75%", "76-99%", "100%"])
por_faixa = s2.groupby("faixa", observed=True)["evadiu"].agg(["mean", "size"]).reset_index()

fig = go.Figure(go.Bar(
    x=por_faixa["faixa"].astype(str), y=por_faixa["mean"] * 100, marker_color=AZUL,
    customdata=por_faixa["size"], text=[f"{v:.0f}%" for v in por_faixa["mean"] * 100], textposition="outside",
    hovertemplate="Aprovação %{x}<br>Evasão: %{y:.1f}%<br>%{customdata} alunos<extra></extra>"))
fig.update_layout(title="Taxa de evasão por aprovação no 2º semestre",
                  xaxis_title="Disciplinas aprovadas no 2º semestre (% das matriculadas)",
                  yaxis_title="Evadiram (%)", yaxis_range=[0, 105], bargap=0.35, height=420)
fig.show()

In [ ]:
por_curso = (df.assign(curso_nome=df["curso"].map(CURSOS))
             .groupby("curso_nome")["evadiu"].agg(["mean", "size"])
             .sort_values("mean").reset_index())
fig = go.Figure(go.Bar(
    y=por_curso["curso_nome"], x=por_curso["mean"] * 100, orientation="h", marker_color=AZUL,
    customdata=por_curso["size"],
    hovertemplate="%{y}<br>Evasão: %{x:.1f}%<br>%{customdata} alunos<extra></extra>"))
fig.update_layout(title="Taxa de evasão por curso", xaxis_title="Evadiram (%)",
                  height=560, bargap=0.3, margin=dict(l=10))
fig.show()

## 3. Engenharia de atributos

Os dados originais dizem *quantas* disciplinas o aluno pegou e *quantas* passou. A função `criar_atributos`
(em `src/dados.py`) cria colunas que traduzem isso para o problema:

| Atributo novo | Como é calculado |
|---|---|
| `s1_taxa_aprovacao`, `s2_taxa_aprovacao` | aprovadas ÷ matriculadas em cada semestre |
| `variacao_nota` | nota média do 2º semestre − nota média do 1º |
| `total_reprovadas` | disciplinas não aprovadas nos dois semestres |
| `total_sem_avaliacao` | disciplinas sem avaliação nos dois semestres |
| `zerou_semestre` | 1 se não passou em nenhuma disciplina em algum semestre |
| `situacao_financeira_ruim` | 1 se é devedor ou está com a mensalidade atrasada |

Essa função faz parte do *pipeline* do modelo, então é aplicada automaticamente no treino e na previsão.

In [ ]:
novas = ["s1_taxa_aprovacao", "s2_taxa_aprovacao", "variacao_nota", "total_reprovadas",
         "total_sem_avaliacao", "zerou_semestre", "situacao_financeira_ruim"]
exemplo = criar_atributos(df.sample(5, random_state=1))
exemplo[["s1_matriculadas", "s1_aprovadas", "s2_matriculadas", "s2_aprovadas"] + novas].round(2)

## 4. Treino

O script `treinar.py` faz o processo completo:

1. **Separa 20% dos alunos para teste** logo no início. Eles só são usados na avaliação final.
2. Compara **Regressão Logística**, **Random Forest** e **XGBoost** com **validação cruzada estratificada de 5 partes**
   nos outros 80%. O desbalanceamento é tratado com `class_weight` / `scale_pos_weight`.
3. Escolhe o modelo com **maior ROC-AUC**.
4. Escolhe **dois limiares** na validação cruzada:
   - 🔴 **Alto**: maximiza o **F1** (equilíbrio, poucos alarmes falsos)
   - 🟡 **Médio**: maximiza o **F2** (dá mais peso ao recall, para quase nenhum evasor passar despercebido)
5. Avalia no teste, calcula a importância dos atributos e salva tudo em `modelos/`.

Leva cerca de 1 minuto.

In [ ]:
!uv run python treinar.py

## 5. Resultados

In [ ]:
m = json.load(open("modelos/metricas.json", encoding="utf-8"))
comp = pd.DataFrame(m["comparacao_cv"]).T
comp.columns = ["ROC-AUC", "Acurácia", "Precisão", "Recall", "F1"]
print("Validação cruzada 5-fold (treino, limiar 0,5):")
display(comp.round(3))
print(f"Modelo escolhido: {m['modelo_escolhido']}")

In [ ]:
linhas = []
for chave, nome in [("alto", "Alto (máx. F1)"), ("medio", "Médio (máx. F2)")]:
    r = m["teste"][chave]
    linhas.append({"limiar": f"{nome}: {m['limiares'][chave]:.0%}", "Acurácia": r["acuracia"],
                   "Precisão": r["precisao"], "Recall": r["recall"], "F1": r["f1"]})
print(f"Conjunto de teste: {m['n_teste']} alunos | ROC-AUC = {m['teste']['alto']['roc_auc']:.3f}")
pd.DataFrame(linhas).set_index("limiar").round(3)

- **Precisão**: dos alunos que o modelo alertou, quantos realmente evadiram.
- **Recall**: dos alunos que realmente evadiram, quantos o modelo alertou.

In [ ]:
from plotly.subplots import make_subplots

rotulos = ["Não evadiu", "Evadiu"]
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.15, subplot_titles=[
    f"Limiar Alto ({m['limiares']['alto']:.0%})", f"Limiar Médio ({m['limiares']['medio']:.0%})"])
for i, chave in enumerate(["alto", "medio"], start=1):
    mc = m["teste"][chave]["matriz_confusao"]
    fig.add_trace(go.Heatmap(
        z=mc, x=rotulos, y=rotulos, colorscale=[[0, "#cde2fb"], [1, "#184f95"]], showscale=False,
        text=mc, texttemplate="%{text}", textfont=dict(size=18), xgap=2, ygap=2,
        hovertemplate="Real: %{y}<br>Previsto: %{x}<br>%{z} alunos<extra></extra>"), row=1, col=i)
    fig.update_xaxes(title_text="Previsto", row=1, col=i)
    fig.update_yaxes(title_text="Real", autorange="reversed", row=1, col=i)
fig.update_layout(title="Matrizes de confusão no teste", height=380)
fig.show()

In [ ]:
fig = go.Figure()
fig.add_scatter(x=[0, 1], y=[0, 1], mode="lines", name="Aleatório (AUC 0,5)",
                line=dict(color="#8a8984", dash="dash", width=1), hoverinfo="skip")
fig.add_scatter(x=m["roc"]["fpr"], y=m["roc"]["tpr"], mode="lines",
                name=f"{m['modelo_escolhido']} (AUC {m['teste']['alto']['roc_auc']:.3f})",
                line=dict(color=AZUL, width=2),
                hovertemplate="Falsos positivos: %{x:.1%}<br>Recall: %{y:.1%}<extra></extra>")
fig.update_layout(title="Curva ROC no teste", xaxis_title="Taxa de falsos positivos",
                  yaxis_title="Taxa de verdadeiros positivos (recall)", height=450,
                  legend=dict(x=0.45, y=0.08))
fig.show()

In [ ]:
imp = pd.DataFrame(m["importancias"]).head(12).iloc[::-1]
fig = go.Figure(go.Bar(
    x=imp["importancia"], y=imp["atributo"].map(ROTULOS), orientation="h", marker_color=AZUL,
    error_x=dict(type="data", array=imp["desvio"], color="#8a8984"),
    hovertemplate="%{y}: %{x:.4f}<extra></extra>"))
fig.update_layout(title="Importância por permutação (quanto o ROC-AUC cai ao embaralhar a coluna)",
                  xaxis_title="Queda no ROC-AUC", height=480, bargap=0.35, margin=dict(l=10))
fig.show()

## 6. Usando o modelo

Previsão para alguns alunos fictícios. Os campos que não foram informados usam o valor mais comum da base.
A previsão roda com `uv run`, no mesmo ambiente em que o modelo foi treinado, para garantir as versões certas
do scikit-learn e do XGBoost.

Altere os valores no dicionário `cenarios` e rode de novo para testar outros casos.

In [ ]:
%%writefile prever_cenarios.py
import joblib
import pandas as pd
from src.dados import COLUNAS_ENTRADA, faixa_risco

artefato = joblib.load("modelos/modelo.joblib")
modelo, limiares, ref = artefato["pipeline"], artefato["limiares"], artefato["referencia"]

# Mesmos valores padrão do formulário do app
base = {"turno_diurno": 1, "genero_masculino": 0, "deslocado": 0, "s1_matriculadas": 6, "s1_aprovadas": 5, "s1_nota_media": 12.5,
        "s2_matriculadas": 6, "s2_aprovadas": 5, "s2_nota_media": 12.5,
        "mensalidades_em_dia": 1, "devedor": 0, "bolsista": 0, "s1_sem_avaliacao": 0, "s2_sem_avaliacao": 0}
cenarios = {
    "Aluno típico": {},
    "Aprovou 3 de 6 no 2º semestre": {"s2_aprovadas": 3, "s2_nota_media": 11.0},
    "Aprovou 2 de 6 no 2º semestre": {"s2_aprovadas": 2, "s2_nota_media": 10.0},
    "2 de 6 + mensalidade atrasada": {"s2_aprovadas": 2, "s2_nota_media": 10.0, "mensalidades_em_dia": 0},
    "Só a mensalidade atrasada": {"mensalidades_em_dia": 0},
    "Zerou o 2º semestre": {"s2_aprovadas": 0, "s2_nota_media": 0.0},
}

linhas = []
for nome, mudancas in cenarios.items():
    aluno = {**ref, **base, **mudancas}
    aluno["s1_avaliacoes"] = aluno["s1_matriculadas"] + 2
    aluno["s2_avaliacoes"] = aluno["s2_matriculadas"] + 2
    prob = modelo.predict_proba(pd.DataFrame([aluno])[COLUNAS_ENTRADA].astype(float))[0, 1]
    linhas.append({"cenário": nome, "prob. evasão": f"{prob:.0%}", "risco": faixa_risco(prob, limiares)})
print(pd.DataFrame(linhas).to_string(index=False))

In [ ]:
!uv run python prever_cenarios.py

## 7. Abrir o app (Streamlit)

O app tem o painel da turma, o formulário de avaliação individual e a aba de desempenho do modelo.

- **No Colab:** a célula abaixo inicia o Streamlit e cria um túnel público temporário com o
  [cloudflared](https://github.com/cloudflare/cloudflared). Ela imprime um link `https://....trycloudflare.com`.
  Esse link só funciona enquanto a sessão do Colab estiver ativa.
- **No seu computador** (mais estável): `uv run streamlit run app.py` e abrir http://localhost:8501

In [ ]:
import re, subprocess, time

if IN_COLAB:
    subprocess.Popen("uv run streamlit run app.py --server.port 8501 --server.headless true > streamlit.log 2>&1", shell=True)
    if not os.path.exists("cloudflared"):
        !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
        !chmod +x cloudflared
    subprocess.Popen("./cloudflared tunnel --url http://localhost:8501 > tunnel.log 2>&1", shell=True)

    url = None
    for _ in range(30):
        time.sleep(2)
        achado = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
        if achado:
            url = achado.group(0)
            break
    print(f"App: {url}  (pode levar uns 10 segundos para abrir)" if url else "Túnel não subiu. Veja tunnel.log")
else:
    print("Fora do Colab: rode `uv run streamlit run app.py` no terminal e abra http://localhost:8501")